## REL has DEF

In [1]:
import pandas as pd

umls_def = pd.read_csv('UMLS_DEF.csv')
umls_doc = pd.read_csv('UMLS_DOC.csv')
umls_rel = pd.read_csv('UMLS_REL.csv').drop_duplicates()
umls_def.shape, umls_doc.shape

((244911, 3), (907, 2))

In [2]:
umls_rel = pd.merge(umls_rel, umls_def, left_on='CUI1', right_on='cui', how='left')
umls_rel = pd.merge(umls_rel, umls_def, left_on='CUI2', right_on='cui', how='left')
umls_rel = umls_rel.dropna()
umls_rel

,CUI1,REL,CUI2,cui_x,name_x,definition_x,cui_y,name_y,definition_y
1,C0000039,AQ,C0001555,C0000039,"DPPC, Phosphatidylcholine, Dipalmitoyl, Dipalm...",Synthetic phospholipid used in liposomes and l...,C0001555,"administration & dosage, administration and do...","Used with drugs for dosage forms, routes of ad..."
2,C0000039,AQ,C0001688,C0000039,"DPPC, Phosphatidylcholine, Dipalmitoyl, Dipalm...",Synthetic phospholipid used in liposomes and l...,C0001688,"ADV EFF, side effects, aspects of adverse effe...","Used with drugs, chemicals, or biological agen..."
3,C0000039,AQ,C0002776,C0000039,"DPPC, Phosphatidylcholine, Dipalmitoyl, Dipalm...",Synthetic phospholipid used in liposomes and l...,C0002776,"analogs & derivatives, analogs and derivatives...",Used with drugs and chemicals for substances t...
4,C0000039,AQ,C0003139,C0000039,"DPPC, Phosphatidylcholine, Dipalmitoyl, Dipalm...",Synthetic phospholipid used in liposomes and l...,C0003139,"antagonists & inhibitors, AI, inhibitor/antago...","Used with chemicals, drugs, and endogenous sub..."
5,C0000039,AQ,C0005572,C0000039,"DPPC, Phosphatidylcholine, Dipalmitoyl, Dipalm...",Synthetic phospholipid used in liposomes and l...,C0005572,"BIOSYN, bioformation, anabolism, BI, biosynthesis",Used for the anabolic formation of chemical su...
...,...,...,...,...,...,...,...,...,...
20592989,C5847615,RO,C0443146,C5847615,"Diabetes mellitus type 1 stage 2, Diabetes mel...",Includes individuals with two or more islet au...,C0443146,"Autoimmune, autoimmune responses, Autoimmune R...",A specific humoral or cell-mediated immune res...
20593021,C5847622,PAR,C0011854,C5847622,"Diabetes mellitus type 1 stage 3, Diabetes mel...",Represents manifestations of the typical clini...,C0011854,"Diabetes mellitus - juvenile, Juvenile diabete...","<p><a href=""https://medlineplus.gov/diabetes.h..."
20593022,C5847622,PAR,C0271650,C5847622,"Diabetes mellitus type 1 stage 3, Diabetes mel...",Represents manifestations of the typical clini...,C0271650,"glucose intolerance, Glucose Tolerances, Impai...",The inability to regulate blood glucose levels...
20593025,C5847622,RO,C0014136,C5847622,"Diabetes mellitus type 1 stage 3, Diabetes mel...",Represents manifestations of the typical clini...,C0014136,"Endocrine System, Endocrine system structure, ...",Collective designation for those tissues capab...


#### Statistics

In [3]:
umls_rel.describe()

,CUI1,REL,CUI2,cui_x,name_x,definition_x,cui_y,name_y,definition_y
count,4083088,4083088,4083088,4083088,4083088,4083088,4083088,4083088,4083088
unique,239303,9,239303,239303,239303,231412,239303,239303,231412
top,C1880104,RO,C1880104,C1880104,"CDISC Terminology, CDISC, Clinical Data Interc...",The terminology that includes terms relevant t...,C1880104,"CDISC Terminology, CDISC, Clinical Data Interc...",The terminology that includes terms relevant t...
freq,27346,1890290,27346,27346,27346,27346,27346,27346,27346


## Save unique entities and relations

In [16]:
import pickle
import pandas as pd

umls_rel = pd.read_csv('UMLS_REL+RELA/UMLS_REL_hasDEF_nor20.csv')
unique_items = umls_rel['REL'].unique().tolist()
with open('UMLS_REL+RELA/REL', 'wb') as fp:
    pickle.dump(unique_items, fp)

unique_items1 = umls_rel['CUI1'].unique().tolist()
unique_items2 = umls_rel['CUI2'].unique().tolist()
print(len(unique_items2))
unique_items = list(set(unique_items1 + unique_items2))
print(len(unique_items))
with open('UMLS_REL+RELA/CUI', 'wb') as fp:
    pickle.dump(unique_items, fp)

224043
236066


#### Encode REL

In [17]:
import pickle
import pandas as pd
from tqdm import tqdm

umls_rel = pd.read_csv('UMLS_REL+RELA/UMLS_REL_hasDEF_nor20.csv')
with open('UMLS_REL+RELA/REL', 'rb') as fp:
    relation_list = pickle.load(fp)

for i, row in tqdm(umls_rel.iterrows()):
    row['REL'] = relation_list.index(row['REL'])
umls_rel

1859292it [01:35, 19521.46it/s]


,CUI1,REL,CUI2
0,C0000039,0,C0001555
1,C0000039,0,C0001688
2,C0000039,0,C0002776
3,C0000039,0,C0003139
4,C0000039,0,C0005572
...,...,...,...
1859287,C5828584,2,C1615053
1859288,C5828614,3,C3658315
1859289,C5847614,6,C0014136
1859290,C5847615,6,C0014136


#### Add Negative Samples

In [18]:
import random
from tqdm import tqdm

with open('UMLS_REL+RELA/CUI', 'rb') as fp:
    entity_list = pickle.load(fp)
group = umls_rel.groupby('CUI1')['CUI2'].apply(set).reset_index()

pointer = 0
neg_tail_list = []
for i, row in tqdm(umls_rel.iterrows()):
    # align head
    if row['CUI1'] != group['CUI1'][pointer]:
        pointer += 1
    # randomly choose negative tail
    neg_tail = random.choice(entity_list)
    while neg_tail in group['CUI2'][pointer]:
        neg_tail = random.choice(entity_list)
    # add negative tail to the list
    neg_tail_list.append(neg_tail)

len(neg_tail_list)

1859292it [01:46, 17463.43it/s]


1859292

In [19]:
umls_rel['neg_CUI2'] = neg_tail_list
umls_rel.describe()

,CUI1,REL,CUI2,neg_CUI2
0,C0000039,0,C0001555,C1709108
1,C0000039,0,C0001688,C4687345
2,C0000039,0,C0002776,C5558399
3,C0000039,0,C0003139,C0162758
4,C0000039,0,C0005572,C0281642
...,...,...,...,...
1859287,C5828584,2,C1615053,C0110666
1859288,C5828614,3,C3658315,C0346251
1859289,C5847614,6,C0014136,C1336200
1859290,C5847615,6,C0014136,C4280692


#### Replace CUI with DEF

In [20]:
umls_def = pd.read_csv('UMLS_DEF.csv')[['cui', 'definition']]
umls_rel = pd.merge(umls_rel, umls_def, left_on='CUI1', right_on='cui', how='left')
umls_rel = pd.merge(umls_rel, umls_def, left_on='CUI2', right_on='cui', how='left')
umls_rel = pd.merge(umls_rel, umls_def, left_on='neg_CUI2', right_on='cui', how='left')
umls_rel

,CUI1,REL,CUI2,neg_CUI2,cui_x,definition_x,cui_y,definition_y,cui,definition
0,C0000039,0,C0001555,C1709108,C0000039,Synthetic phospholipid used in liposomes and l...,C0001555,"Used with drugs for dosage forms, routes of ad...",C1709108,A general term that refers to a TNM finding of...
1,C0000039,0,C0001688,C4687345,C0000039,Synthetic phospholipid used in liposomes and l...,C0001688,"Used with drugs, chemicals, or biological agen...",C4687345,Death
2,C0000039,0,C0002776,C5558399,C0000039,Synthetic phospholipid used in liposomes and l...,C0002776,Used with drugs and chemicals for substances t...,C5558399,Abnormally low blood oxygen level without the ...
3,C0000039,0,C0003139,C0162758,C0000039,Synthetic phospholipid used in liposomes and l...,C0003139,"Used with chemicals, drugs, and endogenous sub...",C0162758,Compounds that specifically inhibit the reupta...
4,C0000039,0,C0005572,C0281642,C0000039,Synthetic phospholipid used in liposomes and l...,C0005572,Used for the anabolic formation of chemical su...,C0281642,An acute myeloid leukemia with minimal differe...
...,...,...,...,...,...,...,...,...,...,...
1859287,C5828584,2,C1615053,C0110666,C5828584,"A non-human influenza virus strain, which norm...",C1615053,A subtype of INFLUENZA A VIRUS comprised of th...,C0110666,A derivative of the original MOPP regimen cons...
1859288,C5828614,3,C3658315,C0346251,C5828614,Health effects associated with activities of b...,C3658315,"The conditions in which people are born, grow,...",C0346251,A sarcoma that arises from the kidney.
1859289,C5847614,6,C0014136,C1336200,C5847614,Represents individuals who have developed two ...,C0014136,Collective designation for those tissues capab...,C1336200,"Stage IIC includes: T2c, N0, M0. T2c: Pelvic e..."
1859290,C5847615,6,C0014136,C4280692,C5847615,Includes individuals with two or more islet au...,C0014136,Collective designation for those tissues capab...,C4280692,A deviation in the size of nasopharyngeal aden...


In [21]:
umls_rel = umls_rel[['definition_x', 'REL', 'definition_y', 'definition']]
umls_rel = umls_rel.rename(columns={'definition_x': 'CUI1', 'definition_y': 'CUI2', 'definition': 'neg_CUI2'})
umls_rel

,CUI1,REL,CUI2,neg_CUI2
0,Synthetic phospholipid used in liposomes and l...,0,"Used with drugs for dosage forms, routes of ad...",A general term that refers to a TNM finding of...
1,Synthetic phospholipid used in liposomes and l...,0,"Used with drugs, chemicals, or biological agen...",Death
2,Synthetic phospholipid used in liposomes and l...,0,Used with drugs and chemicals for substances t...,Abnormally low blood oxygen level without the ...
3,Synthetic phospholipid used in liposomes and l...,0,"Used with chemicals, drugs, and endogenous sub...",Compounds that specifically inhibit the reupta...
4,Synthetic phospholipid used in liposomes and l...,0,Used for the anabolic formation of chemical su...,An acute myeloid leukemia with minimal differe...
...,...,...,...,...
1859287,"A non-human influenza virus strain, which norm...",2,A subtype of INFLUENZA A VIRUS comprised of th...,A derivative of the original MOPP regimen cons...
1859288,Health effects associated with activities of b...,3,"The conditions in which people are born, grow,...",A sarcoma that arises from the kidney.
1859289,Represents individuals who have developed two ...,6,Collective designation for those tissues capab...,"Stage IIC includes: T2c, N0, M0. T2c: Pelvic e..."
1859290,Includes individuals with two or more islet au...,6,Collective designation for those tissues capab...,A deviation in the size of nasopharyngeal aden...


In [22]:
umls_rel.to_csv('UMLS_REL+RELA/UMLS_REL_allDEF_nor20.csv', index=False)

In [23]:
umls_rel.describe()

,CUI1,REL,CUI2,neg_CUI2
count,1859292,1859292,1859292,1859292
unique,132256,20,217126,228167
top,The terminology that includes terms relevant t...,0,Used for taxonomic or other systematic or hier...,Death
freq,27345,624581,25160,3643


#### Remove Reverse Relations + Cut Data

In [1]:
import pandas as pd

umls_rel = pd.read_csv('UMLS_REL+RELA/UMLS_REL_hasDEF.csv')
group = umls_rel.groupby('REL').size().reset_index(name='Count').sort_values(by='Count', ascending=False, ignore_index=True)
group

,REL,Count
0,AQ,624581
1,QB,624581
2,concept_in_subset,320620
3,subset_includes_concept,320620
4,isa,317575
...,...,...
675,surgical_approach_of,1
676,larger_than,1
677,procedure_has_imaged_anatomy,1
678,has_specimen_source_morphology,1


In [2]:
rel_list = []
for i in range(0, len(group)):
    if i == 0:
        rel_list.append(group['REL'][i])
        continue
    if group['Count'][i] != group['Count'][i - 1]:
        rel_list.append(group['REL'][i])
len(rel_list)

247

In [3]:
new_umls_rel = umls_rel[umls_rel['REL'].isin(rel_list)]
new_umls_rel.describe()

,CUI1,REL,CUI2
count,2163988,2163988,2163988
unique,162706,247,226582
top,C1880104,AQ,C0008903
freq,27345,624581,25160


In [4]:
new_group = new_umls_rel.groupby('REL').size().reset_index(name='Count').sort_values(by='Count', ascending=False, ignore_index=True)
new_group

,REL,Count
0,AQ,624581
1,concept_in_subset,320620
2,isa,317575
3,RO,114102
4,PAR,72262
...,...,...
242,has_specimen_procedure,5
243,has_owning_affiliate,4
244,chemical_or_drug_metabolism_is_associated_with...,3
245,chemical_or_drug_affects_abnormal_cell,2


In [14]:
selected_rows = new_umls_rel[new_umls_rel['REL'].isin(new_group['REL'][:20])]
unique_items1 = selected_rows['CUI1'].unique().tolist()
unique_items2 = selected_rows['CUI2'].unique().tolist()
unique_items = list(set(unique_items1 + unique_items2))
print(len(unique_items))
selected_rows.describe()

236066


,CUI1,REL,CUI2
count,1859292,1859292,1859292
unique,136461,20,224043
top,C1880104,AQ,C0008903
freq,27345,624581,25160


In [15]:
selected_rows.to_csv('UMLS_REL+RELA/UMLS_REL_hasDEF_nor20.csv', index=False)

In [13]:
import numpy as np

arr = np.load('weights/KG_000.npy')
arr.shape

(1024, 3, 768)

In [15]:
arr1 = np.load('weights/KG_avg_001.npy')
arr1.shape

(1024, 768)

#### generate posDEF

In [1]:
import pandas as pd
import pickle

with open('pickle_relation', 'rb') as fp:
    relation_list = pickle.load(fp)
data_df = pd.read_csv('UMLS_REL+RELA/UMLS_REL_allDEF.csv')[['CUI1', 'REL', 'CUI2']]
doc_df = pd.read_csv('UMLS_DOC.csv').dropna()
doc_df['EXPL'] = doc_df['EXPL'].str.replace('_', ' ')
doc_df['EXPL'] = doc_df['EXPL'].str.lower()

In [2]:
from tqdm import tqdm

for i, row in tqdm(data_df.iterrows()):
    rel = doc_df[doc_df['VALUE'].str.contains(relation_list[row['REL']])].iloc[0]['EXPL']
    data_df.loc[i, 'REL'] = rel
data_df

0it [00:00, ?it/s]C:\Users\wxc\AppData\Local\Temp\ipykernel_27412\585836826.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'allowed qualifier' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data_df.loc[i, 'REL'] = rel
4206228it [41:50, 1675.77it/s]


,CUI1,REL,CUI2
0,Synthetic phospholipid used in liposomes and l...,allowed qualifier,"Used with drugs for dosage forms, routes of ad..."
1,Synthetic phospholipid used in liposomes and l...,allowed qualifier,"Used with drugs, chemicals, or biological agen..."
2,Synthetic phospholipid used in liposomes and l...,allowed qualifier,Used with drugs and chemicals for substances t...
3,Synthetic phospholipid used in liposomes and l...,allowed qualifier,"Used with chemicals, drugs, and endogenous sub..."
4,Synthetic phospholipid used in liposomes and l...,allowed qualifier,Used for the anabolic formation of chemical su...
...,...,...,...
4206223,Includes individuals with two or more islet au...,pathological process of,A specific humoral or cell-mediated immune res...
4206224,Represents manifestations of the typical clini...,inverse is a,"<p><a href=""https://medlineplus.gov/diabetes.h..."
4206225,Represents manifestations of the typical clini...,inverse is a,The inability to regulate blood glucose levels...
4206226,Represents manifestations of the typical clini...,finding site of,Collective designation for those tissues capab...


In [3]:
data_df.to_csv('UMLS_REL+RELA/UMLS_REL_posDEF.csv', index=False)

In [5]:
data_df['CUI1'][:10].tolist()

['Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membra

In [1]:
import pickle

with open('pickle_relation_id', 'rb') as fp:
    relation_id_list = pickle.load(fp)
with open('pickle_entity_id', 'rb') as fp:
    entity_id_list = pickle.load(fp)
print(len(relation_id_list))
print(len(entity_id_list))

680
239303


In [2]:
import pandas as pd

def_df = pd.read_csv('UMLS_DEF.csv')
doc_df = pd.read_csv('UMLS_DOC.csv').dropna()
doc_df['EXPL'] = doc_df['EXPL'].str.replace('_', ' ')
doc_df['EXPL'] = doc_df['EXPL'].str.lower()

In [14]:
from tqdm import tqdm
relation_list, entity_list = [], []
for rel_id in tqdm(relation_id_list):
    rel = doc_df[doc_df['VALUE'].str.contains(rel_id)].iloc[0]['EXPL']
    relation_list.append(rel)

i = 0
for ent_id in tqdm(entity_id_list):
    for j in range(i, len(def_df)):
        if ent_id == def_df['cui'][j]:
            entity_list.append(def_df['definition'][j])
            i = j
            break


100%|██████████| 239303/239303 [00:02<00:00, 88156.81it/s]


In [25]:
with open('pickle_relation', 'wb') as fp:
    pickle.dump(relation_list, fp)
with open('pickle_entity', 'wb') as fp:
    pickle.dump(entity_list, fp)

In [3]:
import pandas as pd
a = pd.read_csv('UMLS_REL+RELA/UMLS_REL_all.csv')
a

,CUI1,REL,CUI2,neg_CUI2
0,C0000039,0,C0001555,C2987321
1,C0000039,0,C0001688,C0243009
2,C0000039,0,C0002776,C1555573
3,C0000039,0,C0003139,C1334239
4,C0000039,0,C0005572,C0999515
...,...,...,...,...
4206223,C5847615,68,C0443146,C4749431
4206224,C5847622,3,C0011854,C1553689
4206225,C5847622,3,C0271650,C0238547
4206226,C5847622,60,C0014136,C1422661


#### retrieve

In [1]:
import numpy as np
import pandas as pd

topk_idx_img = np.load('topk_idx/topk_idx_img_PC_avg_IVFPQ_pca.npy')
topk_idx_que = np.load('topk_idx/topk_idx_que_PC_avg_IVFPQ_pca.npy')
umls = pd.read_csv('UMLS_REL+RELA/UMLS_REL_posDEF.csv')
test = pd.read_csv('../SOURCE/PMC-VQA/test_clean.csv')[['Figure_path', 'Question', 'Answer']]

In [3]:
umls.iloc[0]

CUI1    Synthetic phospholipid used in liposomes and l...
REL                                     allowed qualifier
CUI2    Used with drugs for dosage forms, routes of ad...
Name: 0, dtype: object

In [3]:
concat_tri = umls.apply(lambda row: ' '.join(row), axis=1)

In [7]:
test['top1_img'] = concat_tri[topk_idx_img[:, 0]].reset_index(drop=True)
test['top2_img'] = concat_tri[topk_idx_img[:, 1]].reset_index(drop=True)
test['top3_img'] = concat_tri[topk_idx_img[:, 2]].reset_index(drop=True)
test['top4_img'] = concat_tri[topk_idx_img[:, 3]].reset_index(drop=True)
test['top5_img'] = concat_tri[topk_idx_img[:, 4]].reset_index(drop=True)
test['top1_que'] = concat_tri[topk_idx_que[:, 0]].reset_index(drop=True)
test['top2_que'] = concat_tri[topk_idx_que[:, 1]].reset_index(drop=True)
test['top3_que'] = concat_tri[topk_idx_que[:, 2]].reset_index(drop=True)
test['top4_que'] = concat_tri[topk_idx_que[:, 3]].reset_index(drop=True)
test['top5_que'] = concat_tri[topk_idx_que[:, 4]].reset_index(drop=True)
test

,Figure_path,Question,Answer,top1_img,top2_img,top3_img,top4_img,top5_img,top1_que,top2_que,top3_que,top4_que,top5_que
0,PMC8415802_FIG1.jpg,What is the name of the medical imaging techn...,Magnetic resonance imaging,A mixture of two T-lymphocyte preparations exp...,An emulsion containing the tartrate salt of th...,A recombinant immunotoxin consisting of the Fv...,An emulsion containing the tartrate salt of th...,A preparation of autologous T-lymphocytes that...,A surgical technique to correct REFRACTIVE ERR...,Surgical techniques on the CORNEA employing LA...,"Preparation of TOOTH surfaces, and of material...",The making of a continuous circular tear in th...,The removal of a cataractous CRYSTALLINE LENS ...
1,PMC8415802_FIG1.jpg,What is the appearance of the hyperintense fo...,Hypointense,A mixture of two T-lymphocyte preparations exp...,An emulsion containing the tartrate salt of th...,A recombinant immunotoxin consisting of the Fv...,An emulsion containing the tartrate salt of th...,A preparation of autologous T-lymphocytes that...,"Used with organs, animals, and higher plants a...","Used with organs, animals, and higher plants a...","Used with organs, animals, and higher plants a...","Used with organs, animals, and higher plants a...","Used with organs, animals, and higher plants a..."
2,PMC8415874_f5.jpg,What color did the soft tissue appear as in th...,Green,Surgical techniques on the CORNEA employing LA...,The progressive compaction of dispersed interp...,The removal of DNA LESIONS and/or restoration ...,Practices involved in preventing the transmiss...,The removal of DNA LESIONS and/or restoration ...,"a thin layer of tissue that covers organs, gla...","a thin layer of tissue that covers organs, gla...","a thin layer of tissue that covers organs, gla...","a thin layer of tissue that covers organs, gla...","a thin layer of tissue that covers organs, gla..."
3,PMC8415874_f6.jpg,What color represents the harder area in the ...,Blue,A quantitative microscopic finding indicating ...,A quantitative microscopic finding indicating ...,A semi-quantitative microscopic finding indica...,A semi-quantitative microscopic finding indica...,A semi-quantitative microscopic finding indica...,Cortical malformations secondary to abnormal n...,Congenital structural abnormalities of the DIG...,"The inner, i.e. lumen-facing, lipid bilayer of...",The lipid bilayer surrounding a dense core gra...,The lipid bilayer surrounding any of the compa...
4,PMC8415874_f7.jpg,What is the result of strain elastography in ...,Focal harder (blue) area,A laboratory test result that indicates the pr...,A laboratory test result indicating an abnorma...,A semi-quantitative microscopic finding indica...,A semi-quantitative microscopic finding indica...,A semi-quantitative microscopic finding indica...,antibodies directed against antigenic determin...,A syndrome characterized by the presence of rh...,A strain of PRIMATE T-LYMPHOTROPIC VIRUS 1 iso...,A subtype of molybdenum cofactor deficiency ca...,antibodies directed against antigenic determin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,PMC8161276_toxins-13-00365-f005.jpg,What is depicted in figure (b)?,Location of the sonar probe,"Used with chemicals, biological, and non-biolo...","Used with chemicals, biological, and non-biolo...","Used with chemicals, biological, and non-biolo...","Used with chemicals, biological, and non-biolo...","Used with chemicals, biological, and non-biolo...",Physical location of ETS2_Gene gene has physic...,Human EXTL3 wild-type allele is located within...,Human EXT2 wild-type allele is located within ...,Physical location of EEF1A2_Gene gene has phys...,Physical location of MAFG_Gene gene has physic...
1996,PMC8161291_cancers-13-02490-f006.jpg,What does the inset of the representative flu...,Clusters of mitochondria colocalized with SQST...,"Administration of a liquid, drop by drop, into...",Radioimmunoassay of proteins using antibody co...,"<p>Irrigation, intrag

In [11]:
import json
# Initialize an empty list to store JSON objects
json_data = []

# Iterate over each row in the DataFrame
for index, row in test.iterrows():
    # Create the JSON object
    json_obj = {
        "id": row['Figure_path'].split('.')[0],  # Extracting the image name from the path
        "image": row['Figure_path'],
        "conversations": [
            {
                "from": "human",
                "value": "The retrieved background Knowledge is given as follows: 1." + row['top1_que'].strip() + " 2." + row['top2_que'].strip() + " 3." + row['top1_img'].strip() + " 4." + row['top2_img'].strip() + " The question is: "+ row['Question'].strip()
            },
            {
                "from": "gpt",
                "value": row['Answer'].strip()
            }
        ]
    }
    # Append the JSON object to the list
    json_data.append(json_obj)

# Write the list of JSON objects to a JSON file
with open('llava_data/test_clean_top2.json', 'w') as f:
    json.dump(json_data, f, indent=2)

In [2]:
import pandas as pd

rel = pd.read_csv('UMLS_REL+RELA/UMLS_REL_hasDEF.csv')
defi = pd.read_csv('UMLS_DEF.csv').dropna()
defi

,cui,name,definition
0,C0000039,"DPPC, Phosphatidylcholine, Dipalmitoyl, Dipalm...",Synthetic phospholipid used in liposomes and l...
1,C0000052,"1,4 alpha Glucan Branching Enzyme, Amylo-(1,4-...","In glycogen or amylopectin synthesis, the enzy..."
2,C0000084,"3-Amino-1,1,3-propanetricarboxylic Acid, gamma...","Found in various tissues, particularly in four..."
3,C0000096,"1H-Purine-2,6-dione, 3,7-dihydro-1-methyl-3-(2...",A potent cyclic nucleotide phosphodiesterase i...
4,C0000097,"methylphenyltetrahydropyridine, N-Methyl-4-phe...",A dopaminergic neurotoxic compound which produ...
...,...,...,...
244906,C5847541,"Intervertebral somatic dysfunction type 1, Typ...",A group dysfunction of the thoracic and lumbar...
244907,C5847542,"Type II Intervertebral somatic dysfunction, In...",A single segmental dysfunction of the thoracic...
244908,C5847614,"Diabetes mellitus type 1 stage 1, Diabetes mel...",Represents individuals who have developed two ...
244909,C5847615,"Diabetes mellitus type 1 stage 2, Diabetes mel...",Includes individuals with two or more islet au...


In [16]:
rel[rel['CUI2']=='C0001629']

,CUI1,REL,CUI2
3094,C0000726,CHD,C0022646
7488,C0000769,QB,C0022646
14165,C0001126,PAR,C0022646
14180,C0001126,finding_site_of,C0022646
33383,C0001625,is_location_of_anatomic_structure,C0022646
...,...,...,...
4196603,C5786796,is_associated_anatomic_site_of,C0022646
4196616,C5786797,is_associated_anatomic_site_of,C0022646
4196629,C5786798,is_associated_anatomic_site_of,C0022646
4196664,C5786800,is_associated_anatomic_site_of,C0022646


In [20]:
defi[defi['cui']=='C0022646']

,cui,name,definition
8708,C0022646,"Kidney structure (body structure), Kidney stru...",Body organ that filters blood for the secretio...


In [22]:
import pickle
with open('UMLS_NAME', 'rb') as fp:
    name_dict = pickle.load(fp)

In [24]:
name_dict['C0000039']

{'def': 'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS.',
 'name': ['Dipalmitoyllecithin',
  '1,2-dipalmitoylphosphatidylcholine',
  'Dipalmitoylphosphatidylcholine (substance)',
  '1,2-Dipalmitoylphosphatidylcholine',
  'Phosphatidylcholine, Dipalmitoyl',
  'Dipalmitoyl Phosphatidylcholine',
  '3,5,9-Trioxa-4-phosphapentacosan-1-aminium, 4-hydroxy-N,N,N-trimethyl-10-oxo-7-((1-oxohexadecyl)oxy)-, inner salt, 4-oxide',
  '1,2-Dihexadecyl-sn-Glycerophosphocholine',
  '1,2 Dihexadecyl sn Glycerophosphocholine',
  '1,2-Dipalmitoyl-Glycerophosphocholine',
  '1,2 Dipalmitoylphosphatidylcholine',
  'DIPALMITOYLPHOSPHATIDYLCHOLINE 0102',
  'DPPC',
  'Dipalmitoylglycerophosphocholine',
  'Dipalmitoylphosphatidylcholine',
  '1,2 Dipalmitoyl Glycerophosphocholine']}

In [1]:
import pandas as pd
doc_df = pd.read_csv('UMLS_DOC.csv').dropna()
doc_df['EXPL'] = doc_df['EXPL'].str.replace('_', ' ')
doc_df['EXPL'] = doc_df['EXPL'].str.lower()

In [2]:
umls_doc = {}
for i, row in doc_df.iterrows():
    umls_doc[row['VALUE']] = row['EXPL']

import pickle
with open('UMLS_REL', 'wb') as fp:
    pickle.dump(umls_doc, fp)

In [6]:
with open('UMLS_REL', 'rb') as fp:
    rel_dict = pickle.load(fp)
for k, v in rel_dict.items():
    print(k)

abnormal_cell_affected_by_chemical_or_drug
abnormality_associated_with_allele
absorbability_of
access_device_used_by
access_instrument_of
access_of
acted_on_by_process
active_ingredient_of
active_metabolites_of
active_moiety_of
activity_of_allele
adheres_to
adjacent_to
adjustment_of
afferent_to
after
alias_of
allele_absent_from_wild-type_chromosomal_location
allele_has_abnormality
allele_has_activity
allele_in_chromosomal_location
allele_plays_altered_role_in_process
allele_plays_role_in_metabolism_of_chemical_or_drug
allelic_variant_of
alternative_of
analyzed_by
analyzes
anatomic_structure_has_location
anatomic_structure_is_physical_part_of
anatomical_entity_observed_in
anatomy_originated_from_biological_process
answer_to_is_sterile
answer_to
anterior_to
anteroinferior_to
anterolateral_to
anteromedial_to
anterosuperior_to
approach_of
archetype_of
arterial_supply_of
articulates_with
associated_disease
associated_etiologic_finding_of
associated_finding_of
associated_function_of
associat

#### Inverted Index

In [3]:
import pandas as pd
from transformers import BertTokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import collections
import re


# Function to remove stopwords
def remove_stopwords_and_punctuations(text):
    # Remove punctuations using regex
    text = re.sub(r'[^\w\s]', '', text)
    # Split the text into words and filter out stopwords
    words = text.split()
    words = [word for word in words if word.lower() not in ENGLISH_STOP_WORDS]
    return ' '.join(words)

# Function to create the inverted index
def create_inverted_index(token_id_lists):
    inverted_index = collections.defaultdict(set)

    for idx, token_ids in enumerate(token_id_lists):
        for token_id in token_ids:
            inverted_index[token_id].add(idx)

    return dict(inverted_index)

In [4]:
# load data
df = pd.read_csv('UMLS_REL+RELA/UMLS_REL_posDEF.csv')[:100]
# combine head, relation and entity
df['combined'] = df.apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
# remove stopwords
df['cleaned'] = df['combined'].apply(remove_stopwords_and_punctuations)
# tokenize
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
df['token_ids'] = df['cleaned'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))
df

,CUI1,REL,CUI2,combined,cleaned,token_ids
0,Synthetic phospholipid used in liposomes and l...,allowed qualifier,"Used with drugs for dosage forms, routes of ad...",Synthetic phospholipid used in liposomes and l...,Synthetic phospholipid used liposomes lipid bi...,"[101, 12553, 6887, 2891, 8458, 10893, 23267, 2..."
1,Synthetic phospholipid used in liposomes and l...,allowed qualifier,"Used with drugs, chemicals, or biological agen...",Synthetic phospholipid used in liposomes and l...,Synthetic phospholipid used liposomes lipid bi...,"[101, 12553, 6887, 2891, 8458, 10893, 23267, 2..."
2,Synthetic phospholipid used in liposomes and l...,allowed qualifier,Used with drugs and chemicals for substances t...,Synthetic phospholipid used in liposomes and l...,Synthetic phospholipid used liposomes lipid bi...,"[101, 12553, 6887, 2891, 8458, 10893, 23267, 2..."
3,Synthetic phospholipid used in liposomes and l...,allowed qualifier,"Used with chemicals, drugs, and endogenous sub...",Synthetic phospholipid used in liposomes and l...,Synthetic phospholipid used liposomes lipid bi...,"[101, 12553, 6887, 2891, 8458, 10893, 23267, 2..."
4,Synthetic phospholipid used in liposomes and l...,allowed qualifier,Used for the anabolic formation of chemical su...,Synthetic phospholipid used in liposomes and l...,Synthetic phospholipid used liposomes lipid bi...,"[101, 12553, 6887, 2891, 8458, 10893, 23267, 2..."
...,...,...,...,...,...,...
95,"Found in various tissues, particularly in four...",allowed qualifier,Used with drugs and exogenously administered c...,"Found in various tissues, particularly in four...",various tissues particularly bloodclotting pro...,"[101, 2536, 14095, 3391, 2668, 20464, 21325, 3..."
96,"Found in various tissues, particularly in four...",allowed qualifier,Used with endogenous and exogenous substances ...,"Found in various tissues, particularly in four...",various tissues particularly bloodclotting pro...,"[101, 2536, 14095, 3391, 2668, 20464, 21325, 3..."
97,"Found in various tissues, particularly in four...",allowed qualifier,Used for the chemical preparation of molecules...,"Found in various tissues, particularly in four...",various tissues particularly bloodclotting pro...,"[101, 2536, 14095, 3391, 2668, 20464, 21325, 3..."
98,"Found in various tissues, particularly in four...",has parent relationship in a metathesaurus sou...,Derivatives of GLUTAMIC ACID. Included under t...,"Found in various tissues, particularly in four...",various tissues particularly bloodclotting pro...,"[101, 2536, 14095, 3391, 2668, 20464, 21325, 3..."


In [7]:
token_id_lists = df['token_ids'].tolist()
row_wise_inverted_index = create_inverted_index(token_id_lists)
row_wise_inverted_index

{101: {0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99},
 12553: {0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  65},
 6887: {0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  

In [10]:
row_wise_inverted_index[101]

set

In [11]:
df['cleaned'][0]

'Synthetic phospholipid used liposomes lipid bilayers study biological membranes major constituent PULMONARY SURFACTANTS allowed qualifier Used drugs dosage forms routes administration frequency duration administration quantity medication effects factors'

In [12]:
df['combined'][0]

'Synthetic phospholipid used in liposomes and lipid bilayers to study biological membranes. It is also a major constituent of PULMONARY SURFACTANTS. allowed qualifier Used with drugs for dosage forms, routes of administration, frequency and duration of administration, quantity of medication, and the effects of these factors.'

In [14]:
import pickle
with open('inverted_index', 'rb') as fp:
    rel_dict = pickle.load(fp)
rel_dict[11867]

{1540104,
 163856,
 229415,
 1376314,
 589893,
 1867868,
 1867869,
 1867870,
 1867871,
 1867872,
 1867873,
 1867874,
 99,
 1474693,
 229520,
 229521,
 3211430,
 3211431,
 3211432,
 2687171,
 2687172,
 2687173,
 2687174,
 2687175,
 2687176,
 3408114,
 1376499,
 1376500,
 1376501,
 1376502,
 3408119,
 3408120,
 3408121,
 3408122,
 3408115,
 3408116,
 3408118,
 3408123,
 3408124,
 3408128,
 3408129,
 3408130,
 3408131,
 3408132,
 3408133,
 3408125,
 3408126,
 3408127,
 3080458,
 1114402,
 3342645,
 1671503,
 65872,
 1671504,
 1671505,
 1671506,
 1671507,
 1671508,
 1671509,
 1671510,
 1671511,
 1671512,
 1671514,
 1671515,
 1671516,
 1671517,
 1671513,
 262494,
 1671518,
 1671519,
 983411,
 819583,
 2687450,
 3047904,
 983525,
 2261562,
 2261563,
 2261564,
 262745,
 3572345,
 4031114,
 3408542,
 3408543,
 3408544,
 3048105,
 1344171,
 1344172,
 1344173,
 1344174,
 1344175,
 1344176,
 1344177,
 1344178,
 1344179,
 1344180,
 1344181,
 1344182,
 1344183,
 1344184,
 1344185,
 1344186,
 134418

In [20]:
list(rel_dict.keys())

[6830,
 14608,
 2251,
 11971,
 4486,
 21992,
 2161,
 3933,
 5726,
 3044,
 18690,
 5352,
 24880,
 5563,
 3159,
 9872,
 4195,
 9998,
 4952,
 14057,
 4183,
 3528,
 4392,
 9988,
 6093,
 2564,
 2790,
 10164,
 4820,
 8131,
 3841,
 22709,
 4461,
 2791,
 10070,
 11214,
 4785,
 4272,
 14847,
 16111,
 9323,
 5246,
 5021,
 4548,
 4869,
 10054,
 7948,
 6162,
 5388,
 2551,
 7703,
 3170,
 2129,
 2459,
 6636,
 9030,
 4206,
 2487,
 4399,
 26300,
 3322,
 4703,
 2210,
 8655,
 5355,
 3579,
 20806,
 2856,
 23219,
 3337,
 7373,
 5818,
 2094,
 11309,
 6872,
 2990,
 2333,
 2877,
 4981,
 2840,
 2573,
 4376,
 29969,
 2375,
 13968,
 3223,
 2475,
 2316,
 10611,
 1941,
 3574,
 25442,
 22442,
 15448,
 5466,
 13700,
 5614,
 10663,
 5492,
 3771,
 7029,
 6082,
 5198,
 7845,
 4280,
 5921,
 15382,
 5453,
 17000,
 3531,
 15429,
 1999,
 10072,
 3299,
 4859,
 11896,
 6503,
 3272,
 4039,
 3485,
 12523,
 5310,
 12671,
 3087,
 17383,
 4802,
 6241,
 10520,
 23583,
 13642,
 1019,
 21573,
 1036,
 11125,
 7358,
 4046,
 13729,
 4

In [21]:
(0, 1) == False

False

#### RELA

In [1]:
import pandas as pd

umls = pd.read_csv('UMLS_REL+RELA/UMLS_REL_hasDEF.csv')
umls

,CUI1,REL,CUI2
0,C0000039,AQ,C0001555
1,C0000039,AQ,C0001688
2,C0000039,AQ,C0002776
3,C0000039,AQ,C0003139
4,C0000039,AQ,C0005572
...,...,...,...
4206223,C5847615,pathological_process_of,C0443146
4206224,C5847622,inverse_isa,C0011854
4206225,C5847622,inverse_isa,C0271650
4206226,C5847622,finding_site_of,C0014136


In [3]:
doc = pd.read_csv('UMLS_DOC.csv')[893:-1]
doc['VALUE'].tolist()

['AQ',
 'CHD',
 'DEL',
 'PAR',
 'QB',
 'RB',
 'RN',
 'RO',
 'RQ',
 'RU',
 'SUBX',
 'SY',
 'XR']

In [4]:
strings_to_check = doc['VALUE'].tolist()
mask = umls['REL'].str.contains('|'.join(strings_to_check))
filtered_df = umls[~mask]
filtered_df

,CUI1,REL,CUI2
30,C0000039,mapped_to,C0216971
31,C0000039,inverse_isa,C0031610
32,C0000039,same_as,C0216971
64,C0000052,inverse_isa,C0019495
66,C0000052,exhibited_by,C2256711
...,...,...,...
4206223,C5847615,pathological_process_of,C0443146
4206224,C5847622,inverse_isa,C0011854
4206225,C5847622,inverse_isa,C0271650
4206226,C5847622,finding_site_of,C0014136


In [5]:
import pickle
with open('UMLS_NAME', 'rb') as fp:
    name_dict = pickle.load(fp)
with open('UMLS_REL', 'rb') as fp:
    rel_dict = pickle.load(fp)

from tqdm import tqdm
for i, row in tqdm(filtered_df.iterrows()):
    row['CUI1'] = name_dict[row['CUI1']]['name'][0]
    row['REL'] = rel_dict[row['REL']]
    row['CUI2'] = name_dict[row['CUI2']]['name'][0]
filtered_df

2592580it [02:49, 15273.30it/s]


,CUI1,REL,CUI2
30,Dipalmitoyllecithin,mapped to,colfosceril palmitate
31,Dipalmitoyllecithin,inverse is a,"Acid, Phosphatidic"
32,Dipalmitoyllecithin,same as,colfosceril palmitate
64,"1,4-alpha-D-Glucan:1,4-alpha-D-glucan 6-alpha-...",inverse is a,Hexosyltransferase
66,"1,4-alpha-D-Glucan:1,4-alpha-D-glucan 6-alpha-...",exhibited by,glycogen branching enzyme activity
...,...,...,...
4206223,Diabetes mellitus type 1 stage 2,pathological process of,Autoimmune reaction (finding)
4206224,Diabetes mellitus type 1 stage 3,inverse is a,Type I diabetes mellitus
4206225,Diabetes mellitus type 1 stage 3,inverse is a,diabetes chemical
4206226,Diabetes mellitus type 1 stage 3,finding site of,"Organ System, Endocrine/Metabolic"


In [8]:
filtered_df.to_csv('UMLS_REL+RELA/UMLS_RELA_posDEF.csv', index=False)

In [16]:
filtered_df[filtered_df['REL'] == 'is a']

,CUI1,REL,CUI2
215,alpha Naphthylamine,is a,2-naphthylamine
367,17-Hydroxycorticosteroids,is a,20alpha-Hydroxypregn-4-Ene-3-One
368,17-Hydroxycorticosteroids,is a,Pregnanetriol
369,17-Hydroxycorticosteroids,is a,17-alpha-hydroxyprogesterone
370,17-Hydroxycorticosteroids,is a,17-alpha-hydroxypregnanolone
...,...,...,...
4204162,Abnormal play,is a,Reduced imaginative play skills
4204164,Reduced imaginative play skills,is a,Reduced collaborative imaginative play
4204166,Nonfunctional or atypical use of objects in play,is a,Persistent preoccupation with parts of objects
4204172,Complete cleft of the upper lip,is a,Unilateral complete cleft lip (situation)


In [17]:
filtered_df[filtered_df['REL'] == 'inverse is a']

,CUI1,REL,CUI2
31,Dipalmitoyllecithin,inverse is a,"Acid, Phosphatidic"
64,"1,4-alpha-D-Glucan:1,4-alpha-D-glucan 6-alpha-...",inverse is a,Hexosyltransferase
126,Isobutyltheophylline,inverse is a,Product containing phosphodiesterase inhibitor
151,MPTP,inverse is a,Dopamine Drugs
152,MPTP,inverse is a,Neurotoxins
...,...,...,...
4206217,Diabetes mellitus type 1 stage 1,inverse is a,Type I diabetes mellitus
4206220,Diabetes mellitus type 1 stage 2,inverse is a,Type I diabetes mellitus
4206221,Diabetes mellitus type 1 stage 2,inverse is a,diabetes chemical
4206224,Diabetes mellitus type 1 stage 3,inverse is a,Type I diabetes mellitus


In [19]:
import torch

a = torch.tensor([1, 2, 4], device='cuda')

filtered_df.iloc[a]

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [3]:
import numpy as np

I = np.load(f'PC_IPCA32_IVF_I10.npy').astype(np.int_)
D = np.load(f'PC_IPCA32_IVF_D10.npy')
I

array([[1805112, 3004680, 3675235, ..., 1805136, 3675196, 1727094],
       [1805112, 3004680, 3675235, ..., 1805136, 3675196, 1727094],
       [3913302, 3913303, 3020505, ..., 2387122,  839816, 1858945],
       ...,
       [3004664, 3004665, 1805091, ..., 4089421, 4089422, 1805120],
       [4014148, 2229574, 3051306, ..., 4014154, 3050288, 2813108],
       [ 521705,  568998,  429458, ..., 4013652, 3527580, 3527578]])

In [4]:
topk_idx_img = np.load(f'topk_idx_avg/topk_idx_img_PCTH_avg_HNSW.npy')
topk_idx_img

array([[ 721721,  721727, 2007427, ...,  721728, 1992983, 1993024],
       [ 721721,  721727, 2007427, ...,  721728, 1992983, 1993024],
       [3612265, 3361821, 3612340, ..., 3497578, 3612413, 3612319],
       ...,
       [ 525546, 1017410,  493166, ...,  648742,  486399,   73414],
       [3954175, 3451883, 3448507, ..., 4182122, 3787967, 3443937],
       [3898598, 4001548, 1524550, ..., 1623103, 3676946, 3016556]],
      dtype=int64)

In [38]:
import pandas as pd

df = pd.read_csv('UMLS_REL+RELA/UMLS_RELA_posDEF.csv', keep_default_na=False)
df.shape

(2592580, 3)

In [39]:
df[df['CUI1'] == 'Dose Form']

,CUI1,REL,CUI2
115095,Dose Form,use,capsules
115096,Dose Form,use,salve
115097,Dose Form,use,drug suppository
115098,Dose Form,use,Tablet dose form (qualifier value)
115099,Dose Form,use,Inh
115100,Dose Form,use,pill product
115101,Dose Form,is a,Bath additive (product)
115102,Dose Form,is a,Generator
115103,Dose Form,is a,Kit Dose Form
115104,Dose Form,is a,Unassigned Dose Form


In [26]:
df_cleaned = df[~df.map(lambda x: 'NA' == str(x)).any(axis=1)]
df_cleaned.shape

C:\Users\wxc\AppData\Local\Temp\ipykernel_7304\2806661452.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_cleaned = df[~df.applymap(lambda x: 'NA' == str(x)).any(axis=1)]


(2592558, 3)

In [27]:
df_cleaned[df_cleaned['CUI1'] == 'Dose Form']

,CUI1,REL,CUI2
115095,Dose Form,use,capsules
115096,Dose Form,use,salve
115097,Dose Form,use,drug suppository
115098,Dose Form,use,Tablet dose form (qualifier value)
115099,Dose Form,use,Inh
115100,Dose Form,use,pill product
115101,Dose Form,is a,Bath additive (product)
115102,Dose Form,is a,Generator
115103,Dose Form,is a,Kit Dose Form
115104,Dose Form,is a,Unassigned Dose Form


#### STAT

In [5]:
import nltk
import pandas as pd
# Define stopwords
stop_words = set(nltk.corpus.stopwords.words('english'))

# Function to compare two strings
def compare_strings(text, answer):
    # Tokenize strings into words
    text_words = set(nltk.word_tokenize(text.lower()))
    answer_words = set(nltk.word_tokenize(answer.lower()))

    # Remove stopwords from answer words
    answer_words_without_stopwords = answer_words - stop_words

    # Check if all remaining words in answer appear in text
    if answer_words_without_stopwords.issubset(text_words):
        return True
    else:
        return False

# Function to check if answer is a substring of text
def is_substring(answer, text):
    return answer in text

df = pd.read_csv('../SOURCE/PMC-VQA/test_clean.csv')
# jsonl = pd.read_json('llava_json_data/PC_avg_IVFPQ/test_clean_answer_top1_img.jsonl', lines=True)
jsonl = pd.read_json('llava_json_data/test_clean_answer_32top3_img.jsonl', lines=True)
# jsonl = pd.read_json('llava_json_data/test-clean-answer.jsonl', lines=True)
jsonl['answer'] = df['Answer']
jsonl['question'] = df['Question']

js = pd.read_json('llava_json_data/test-clean-answer.jsonl', lines=True)
js['answer'] = df['Answer']
js['question'] = df['Question']

In [60]:
cl = []
# keywords = ['cancer', 'tumor', 'tumour', 'leukemi', 'carcino', 'lymphoma', 'glioma', 'astrocytoma', 'adenoma',
#             'cholangiocarcinoma', 'chondrosarcoma', 'ependymoma', 'sarcoma', 'melanoma', 'mesothelioma', 'myeloma',
#             'neoplas', 'blastoma', 'meningioma', 'thymoma', 'papilloma', 'pheochromocytoma']

# keywords = ['artery', 'vein', 'blood', 'platelet', 'erythrocyte']

# keywords = ['immune',
#             'marrow', 'thymus', 'lymph', 'spleen', 'tonsil',
#             'leukocyte', 'neutrophil', 'eosinophil', 'basophil', 'monocyte', 'macrophage', 'dendritic cell', 'microglia cell', 'B cell', 'T cell', 'antibod', 'antigen',
#             'complement ', 'cytokines ', 'HLA']

# keywords = ['mri', 'mri scan', 'mri image', 'mri images', 'conventional mri', 'magnetic resonance imaging', 'magnetic resonance imaging (mri)',
#             't1-weighted', 't1-weighted image', 't1-weighted images', 't2-weighted', 't2-weighted image', 't2-weighted images',
#             'diffusion-weighted', 'diffusion-weighted image', 'diffusion-weighted images',
#             'flair', 'flair image', 'flair images',
#             ]

keywords = [' ct', ' ct ', ' ct.', ' ct scan', ' ct image', 'ct images', 'computed tomography', '3dct', '3dct image', '3dct images', 'cat scan', ' ctca ', ' cbct ']

# keywords = ['x-ray', 'x-rays', 'x-ray image', 'x-ray images', 'chest x-ray',]

# keywords = ['ultrasound', 'ultrasound image', 'ultrasound images', 'ultrasonography', 'ultrasound (us)', 'color doppler']

# keywords = ['sem', 'sem image', 'sem images',
#             'tem', 'tem image', 'tem images', 'hr-tem', 'hr-tem image', 'hr-tem images']

cl.append(jsonl[jsonl['question'].str.contains(f'(?:{"|".join(keywords)})', case=False)])
cl.append(jsonl[jsonl['answer'].str.contains(f'(?:{"|".join(keywords)})', case=False)])
cl.append(jsonl[jsonl['answer'].str.contains('^ct scan$', case=False)])

new = pd.concat(cl).sort_index()
new

,question_id,prompt,text,answer_id,model_id,metadata,answer,question
13,PMC8415998_F1,The retrieved background knowledge is given as...,Assistant: The imaging technique used in this ...,Qq8VPCrtEdxniykZVNwUNC,/scratch/xwu20/LLMs/llava-med,{},Computed tomography,What imaging technique was used in this study?
15,PMC8416005_F2,The retrieved background knowledge is given as...,Assistant: The abdominal CT scan showed no abn...,nnX9pSmhL3tCt4iCZRubV5,/scratch/xwu20/LLMs/llava-med,{},Acute hepatitis,What was the result of the abdominal CT in op...
16,PMC8416005_F2,The retrieved background knowledge is given as...,Assistant: The abdominal CT image revealed hyd...,WXbTe2ErwpeujhqMvSh2wQ,/scratch/xwu20/LLMs/llava-med,{},Acute hepatitis,What did the abdominal CT reveal in option A?
19,PMC8416045_F9,The retrieved background knowledge is given as...,Assistant: The figure displays MRI images of t...,QzpumzogVV85QFcdt9uJTt,/scratch/xwu20/LLMs/llava-med,{},MRI and CT,What kind of images are displayed on the left...
44,PMC8416487_FIG3,The retrieved background knowledge is given as...,Assistant: The imaging modality used to obtain...,8mdEioP7roDTbqBTHg83DZ,/scratch/xwu20/LLMs/llava-med,{},CT scan,What imaging modality was used to obtain these...
...,...,...,...,...,...,...,...,...
1869,PMC8927646_F2,The retrieved background knowledge is given as...,Assistant: The purple arrow on the CT image at...,UEVxVA5xSo7nqv7g7zV8LF,/scratch/xwu20/LLMs/llava-med,{},Inflammatory changes in the descending part of...,What did the purple arrow on the CT image ind...
1873,PMC8928079_f0015,The retrieved background knowledge is given as...,Assistant: The imaging technique used in this ...,Ax9BqShJNrCMLkwF9agP5P,/scratch/xwu20/LLMs/llava-med,{},CT scan,What is the name of the imaging technique used...
1890,PMC8930273_fig2,The retrieved background knowledge is given as...,Assistant: The imaging technique used for this...,fji5mB8SXLaeAy6TxJ8o47,/scratch/xwu20/LLMs/llava-med,{},CT scan,What imaging technique was used for this image?
1937,PMC8157460_Fig1,The retrieved background knowledge is given as...,Assistant: The imaging modality used to obtain...,WVkd2orNiM3s9uFswgWSfC,/scratch/xwu20/LLMs/llava-med,{},CT scan,Which imaging modality was used to obtain imag...


In [63]:
# Iterate over each row of the DataFrame
count = 0
idx_list = []
for i, row in new.iterrows():
    text = row['text'].lower()
    answer = row['answer'].lower()
    if compare_strings(text, answer):
        count += 1
        idx_list.append(i)

print(count / len(new) * 100)
print(len(idx_list))

18.064516129032256
28


In [79]:
check = new.loc[idx_list]
i = 15
print(check.loc[idx_list[i], 'prompt'])
print(check.loc[idx_list[i], 'answer'])

The retrieved background knowledge is given as follows: ether degradation is a diorcinol breakdown. down-regulation of cell wall biogenesis, MAPKKK cascade negatively regulated by cell integrity MAPK pathway. positive regulation of cellular component biogenesis is a upregulation of heterochromatin formation. Cell Movement Process gene product plays role in biological process PIERCE2 protein, human. Please answer the question: What could be the cause of the lateral displacement seen in the CT scan?
<image>
Brain injury


In [6]:
# Iterate over each row of the DataFrame
count = 0
jsonl_list = []
for i, row in jsonl.iterrows():
    text = row['text'].lower()
    answer = row['answer'].lower()
    if compare_strings(text, answer):
        count += 1
        jsonl_list.append(i)

print(count / len(jsonl) * 100)
print(len(jsonl_list))

13.5
270


In [7]:
# Iterate over each row of the DataFrame
count = 0
js_list = []
for i, row in js.iterrows():
    text = row['text'].lower()
    answer = row['answer'].lower()
    if compare_strings(text, answer):
        count += 1
        js_list.append(i)

print(count / len(js) * 100)
print(len(js_list))

14.499999999999998
290


In [8]:
difference = [item for item in jsonl_list if item not in js_list]
difference

[19,
 20,
 27,
 28,
 86,
 117,
 123,
 153,
 157,
 182,
 190,
 194,
 209,
 213,
 221,
 291,
 381,
 386,
 444,
 487,
 494,
 530,
 544,
 674,
 682,
 706,
 768,
 778,
 834,
 836,
 847,
 852,
 891,
 927,
 968,
 974,
 991,
 1020,
 1027,
 1050,
 1054,
 1157,
 1164,
 1190,
 1195,
 1202,
 1204,
 1230,
 1236,
 1250,
 1289,
 1325,
 1342,
 1351,
 1389,
 1402,
 1415,
 1417,
 1498,
 1504,
 1514,
 1522,
 1542,
 1569,
 1579,
 1585,
 1608,
 1625,
 1654,
 1657,
 1667,
 1697,
 1728,
 1732,
 1744,
 1762,
 1830,
 1891,
 1892,
 1911,
 1919,
 1924,
 1926,
 1964,
 1965]

In [25]:
check = jsonl.loc[difference]
i = 13
print(check.loc[difference[i], 'question_id'])
print(check.loc[difference[i], 'prompt'])
print(check.loc[difference[i], 'text'])
print(check.loc[difference[i], 'answer'])

PMC8422719_Fig3
The retrieved background knowledge is given as follows: Right Salpingo-Oophorectomy excised anatomy has procedure Fallopian tube, NOS. Lipoma of Spermatic Cord associated morphology of Tumors. Primitive reflexes (palmomental, snout, glabellar) is a Persistent nutritive suckle swallow. Please answer the question: What was the growth pattern of the tumor cells observed on the H&E stain?
<image>
Assistant: The growth pattern of the tumor cells observed on the H&E stain was not predominantly papillary. Instead, the tumor formed an exophytic mass of papillary-like projections.
Papillary growth pattern
